# 07.2. Pipeline: ensemble, temporal rescoring, final submission

1. **Summary of all models** (out-of-fold and fold 0).
2. **Ensembles** of the best models of different families, each in its inference variant from 07.1, fused by
   Weighted Boxes Fusion with weights, on held-out fold 0, where all fold-0 models are unseen.
3. **Temporal rescoring:** a box confirmed in the neighbouring frames of the clip keeps its score, a box seen
   once loses part of it.
4. **Submission:** every member as the ensemble of its four fold models, then the chosen cross-model fusion.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import json

import pandas as pd
import torch

from core import N_FOLDS
from core.cv import best_per_family, compare, environment, existing
from core.data import assign_folds, every_nth, find_data_root, image_key, load_pool
from core.evaluate import coco_evaluate, per_clip_map
from core.submission import write_submission
from core.temporal import rescore
from core.variants import collect_view, make_detector, merge_views, variant_dets

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
frames["fold"] = frames["video"].map(assign_folds(frames))
held0 = every_nth(frames[frames["fold"] == 0], 5)  # fold-0 models never saw these frames
print(len(held0), "held-out frames of fold 0,", held0["video"].nunique(), "clips")

## Summary of all models
Fold 0: every experiment. Out-of-fold: the runs with all folds (outputs of 02–06 and 07.0–07.1 attached).

In [ ]:
summary = pd.read_csv(existing("inference_summary.csv", WORK), index_col=0)
CHOSEN = json.loads(existing("chosen_variants.json", WORK).read_text())
ALL = sorted({f.name[len("metrics_"):-len("_fold0.json")] for root in (WORK, Path("/kaggle/input")) if root.exists()
              for f in root.rglob("metrics_*_fold0.json")})
display(compare(ALL, WORK, fold=0).sort_values("mAP", ascending=False).round(4))
compare(list(best_per_family(["baseline", "yolo_", "rtdetr", "frcnn", "retina", "fcos"], WORK).values()), WORK).sort_values("mAP", ascending=False).round(4)

## Ensembles on fold 0
Members in order of their fold-0 mAP with the chosen variant; the best one can get a higher weight. An ensemble
is kept only if it beats the best single model at both resolutions and in most clips.

In [ ]:
dets = {n: pd.read_parquet(existing(f"fold0_dets_{n}.parquet", WORK)) for n in summary.index}
dets_low = {n: pd.read_parquet(existing(f"fold0_dets_{n}_lowres.parquet", WORK)) for n in summary.index}
best = summary.index[0]
best_clips = pd.Series(per_clip_map(dets[best], held0, gt))


def fuse(members, weights, iou, low=False):
    src = dets_low if low else dets
    return merge_views([src[n] for n in members], iou, "wbf", weights)


OPTIONS = {f"top {k}, weight {w}, iou {iou}": (list(summary.index[:k]), [w] + [1] * (k - 1), iou)
           for k in range(2, len(summary) + 1) for w in (1, 2) for iou in (0.55, 0.7)}
rows = {f"single: {best}": {"mAP": summary.loc[best, "mAP"], "mAP_960": summary.loc[best, "mAP_960"]}}
for label, (members, weights, iou) in OPTIONS.items():
    native = fuse(members, weights, iou)
    rows[label] = {"members": "+".join(members), "mAP": coco_evaluate(native, held0, gt)["mAP"],
                   "mAP_960": coco_evaluate(fuse(members, weights, iou, low=True), held0, gt)["mAP"],
                   "clips_better": int((pd.Series(per_clip_map(native, held0, gt)) > best_clips).sum())}
ens = pd.DataFrame(rows).T
ens.to_csv(WORK / "ensemble_summary.csv")
ens

In [ ]:
cand = ens.loc[list(OPTIONS)]
cand = cand[(cand["mAP"] > summary.loc[best, "mAP"]) & (cand["mAP_960"] > summary.loc[best, "mAP_960"])
            & (cand["clips_better"] > len(best_clips) / 2)]
MEMBERS, WEIGHTS, IOU = OPTIONS[cand["mAP"].astype(float).idxmax()] if len(cand) else ([best], [1], None)
print("final members:", MEMBERS, "weights", WEIGHTS, "fusion IoU", IOU)
fold0 = fuse(MEMBERS, WEIGHTS, IOU) if len(MEMBERS) > 1 else dets[best]

## Temporal rescoring on fold 0
`window`: neighbours on each side, `alpha`: share of the score taken from them. Here neighbours are 5 frames apart
(every 5th frame); on the test set, where every frame is used, they are taken 5 frames apart too.

In [ ]:
keys = held0["key"].tolist()
OPTIONS_T = {"none": None, **{f"window {w}, alpha {a}": dict(window=w, alpha=a) for w in (1, 2) for a in (0.3, 0.5)}}
temporal = pd.Series({label: coco_evaluate(fold0 if o is None else rescore(fold0, keys, o["window"], 0.5, o["alpha"]),
                                           held0, gt)["mAP"] for label, o in OPTIONS_T.items()}, name="mAP")
TEMPORAL = OPTIONS_T[temporal.idxmax()]
print("temporal rescoring:", TEMPORAL)
temporal.round(4).to_frame()

### Findings
*(After the run: whether the ensemble beats the best single model and by how much, which weights and IoU,
and whether temporal rescoring helps.)*

## Submission
Every member: its chosen inference variant with each of its four fold models, fused across folds by WBF;
then the members fused with the chosen weights, then temporal rescoring if it helped on fold 0.
This is the slowest step: views × folds × members passes over the 3.8k test frames.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
test_paths = [str(DATA / f) for f in test["file_name"]]
members = []
for name in MEMBERS:
    folds = []
    for k in range(N_FOLDS):
        detect = make_detector(name, k, WORK)
        views = {v: collect_view(detect, test_paths, v) for v in CHOSEN[name]["views"]}
        folds.append(variant_dets(views, CHOSEN[name]))
        del detect
        torch.cuda.empty_cache()
    members.append(merge_views(folds, 0.55, "wbf"))
    members[-1].to_parquet(WORK / f"test_dets_{name}.parquet")  # kept if a later step fails
    print(name, "done")
test_dets = merge_views(members, IOU, "wbf", WEIGHTS) if len(members) > 1 else members[0]
if TEMPORAL:
    test_dets = rescore(test_dets, [image_key(p) for p in test_paths], TEMPORAL["window"], 0.5, TEMPORAL["alpha"],
                        step=5)  # same distance in time as the every-5th-frame neighbours on fold 0
write_submission(test_dets, test, WORK / "submission_final.csv").head()